In [0]:
from pyspark.sql.functions import explode, col

# Read bronze JSON
df_raw = spark.table("nasa_analytics.bronze.neows_raw")

# The raw JSON is nested under "near_earth_objects"
df_exploded = df_raw.select(explode(col("near_earth_objects")).alias("date_objects"))

# Flatten into (date, asteroid_array)
df_datewise = df_exploded.selectExpr("date_objects.key as close_approach_date", "date_objects.value as asteroids")

In [0]:
# Each date has multiple asteroids → explode further
df_asteroids = df_datewise.withColumn("asteroid", explode("asteroids"))

# Select key fields
df_flat = df_asteroids.select(
    col("close_approach_date"),
    col("asteroid.id").alias("asteroid_id"),
    col("asteroid.name").alias("name"),
    col("asteroid.absolute_magnitude_h").alias("magnitude"),
    col("asteroid.estimated_diameter.kilometers.estimated_diameter_min").alias("diameter_min_km"),
    col("asteroid.estimated_diameter.kilometers.estimated_diameter_max").alias("diameter_max_km"),
    col("asteroid.is_potentially_hazardous_asteroid").alias("is_hazardous"),
    explode(col("asteroid.close_approach_data")).alias("approach")
)

df_structured = df_flat.select(
    "close_approach_date",
    "asteroid_id",
    "name",
    "magnitude",
    "diameter_min_km",
    "diameter_max_km",
    "is_hazardous",
    col("approach.close_approach_date").alias("approach_date"),
    col("approach.relative_velocity.kilometers_per_hour").alias("velocity_kph"),
    col("approach.miss_distance.kilometers").alias("miss_distance_km"),
    col("approach.orbiting_body").alias("orbiting_body")
)

In [0]:
# Save structured asteroid approach table
df_structured.write.mode("overwrite").saveAsTable("nasa_analytics.silver.neows_asteroids")

In [0]:
%sql
USE CATALOG nasa_analytics;
SELECT asteroid_id, name, approach_date, miss_distance_km, is_hazardous
FROM silver.neows_asteroids
LIMIT 20;